# Phase A — Reproduction

Load the combined Maitra et al. (2023) snRNA-seq matrix (GSE213982 + GSE144136,
71 donors, ~160k nuclei), attach the authors' cell-type annotations, and prepare
a clean AnnData for QC, normalization, and the downstream CellChat / ML phases.

**Runtime:** `Runtime > Change runtime type > CPU, High-RAM`. Phase A needs RAM,
not GPU.

Reusable functions live in `src/functions.py` (pulled from GitHub by the boot
cell). Data lives on Google Drive, never in git.

## 1. Environment

In [ ]:
# Install packages not preinstalled on Colab
!pip install -q scanpy anndata harmonypy leidenalg python-igraph GEOparse

import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import os

sc.settings.verbosity = 1
print("scanpy", sc.__version__)
print("anndata", ad.__version__)

## 2. Boot — mount Drive, clone repo, import functions

Edit `REPO_URL` to your GitHub username. `%autoreload 2` means edits pulled from
GitHub take effect without restarting the kernel.

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

# --- data paths (Google Drive) ---
PROJECT_ROOT   = '/content/drive/MyDrive/MLCB_team_project'
RAW_DIR        = os.path.join(PROJECT_ROOT, 'data', 'raw')
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'data', 'checkpoints')
for d in (RAW_DIR, CHECKPOINT_DIR):
    os.makedirs(d, exist_ok=True)

# --- code (GitHub) ---
REPO_URL = 'https://github.com/<username>/MLCB_team_project.git'   # <-- EDIT
REPO_DIR = '/content/MLCB_team_project'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
%load_ext autoreload
%autoreload 2
from functions import build_condition_map, load_dataset

print("Raw data  ->", RAW_DIR)
print("Checkpoints ->", CHECKPOINT_DIR)

## 3. Download the combined matrix from GEO

Three supplementary files: the counts matrix (~1.1 GB, genes x cells), the cell
metadata (the `x` column encoding `donor.barcode.broad.fine`), and the gene names.
`wget -c` resumes on interruption, so a Colab timeout never restarts the download
from zero. Already-downloaded files are skipped.

In [ ]:
import subprocess

GSE = 'GSE213982'
gse_dir = os.path.join(RAW_DIR, GSE)
os.makedirs(gse_dir, exist_ok=True)

base = f"https://ftp.ncbi.nlm.nih.gov/geo/series/GSE213nnn/{GSE}/suppl/"
files = [
    f"{GSE}_combined_counts_matrix.mtx.gz",
    f"{GSE}_combined_counts_matrix_cells_columns.csv.gz",
    f"{GSE}_combined_counts_matrix_genes_rows.csv.gz",
]
for fn in files:
    out = os.path.join(gse_dir, fn)
    if not os.path.exists(out) or os.path.getsize(out) == 0:
        print(f"Downloading {fn} ...")
        r = subprocess.run(["wget", "-c", "-O", out, base + fn])
        if r.returncode != 0:
            print(f"  WARNING: wget returned {r.returncode} for {fn}")
    print(f"  {fn}: {os.path.getsize(out)/1e6:.1f} MB")

## 4. Fetch SOFT metadata for the condition map

The cell string encodes donor / cell-type but **not** diagnosis (MDD vs Control).
Diagnosis lives in the GEO SOFT metadata: GSE213982 for females, GSE144136 for
males. These are small (~KB) — only metadata, not the male count matrix (already
inside the combined matrix above).

In [ ]:
import GEOparse

gse_female = GEOparse.get_GEO(geo='GSE213982',
                              destdir=os.path.join(RAW_DIR, 'GSE213982'), silent=True)
gse_male   = GEOparse.get_GEO(geo='GSE144136',
                              destdir=os.path.join(RAW_DIR, 'GSE144136'), silent=True)
print("Female GSMs:", len(gse_female.gsms))
print("Male GSMs:  ", len(gse_male.gsms))

## 5. Load + verify

`build_condition_map` reads diagnosis for all 71 donors. `load_dataset` loads the
matrix (transposed to cells x genes), parses the cell string into metadata
columns, merges the `M24_2` technical replicate onto `M24`, and attaches
sex/condition. Raw counts are preserved in `.layers['counts']`.

The crosstab is the key sanity check — it must match the paper:
**female 20 MDD / 18 Control, male 17 MDD / 16 Control**.

In [ ]:
condition_map = build_condition_map(gse_female, gse_male)
print(f"Condition map: {len(condition_map)} donors (expect 71)")

adata = load_dataset(RAW_DIR, condition_map)
print(adata)

# --- sanity checks ---
print("\nCells:", adata.n_obs, "| Genes:", adata.n_vars)
print("Unique donors:", adata.obs['donor_id'].nunique(), "(expect 71)")

donor_tbl = adata.obs[['donor_id', 'sex', 'condition']].drop_duplicates()
print("\nDonor-level condition x sex (expect F: 20/18, M: 17/16):")
print(pd.crosstab(donor_tbl['sex'], donor_tbl['condition']))

print("\nM24 merged correctly? (M24_2 gone):",
      'M24_2' not in adata.obs['donor_id'].unique())
print("Broad cell types:", sorted(adata.obs['cell_type_broad'].unique()))
print("N fine clusters:", adata.obs['cell_type_fine'].nunique())

## 6. Checkpoint

Save the loaded AnnData to Drive. This is the first checkpoint — if the session
times out later, reload from here instead of re-running download + load.

In [ ]:
ckpt = os.path.join(CHECKPOINT_DIR, 'phaseA_01_loaded.h5ad')
adata.write_h5ad(ckpt)
print("Saved:", ckpt, f"({os.path.getsize(ckpt)/1e6:.1f} MB)")

# To reload later:
# adata = sc.read_h5ad(ckpt)

## 7. Next: QC / filtering

Coming next — per-nucleus QC (mitochondrial %, gene/count thresholds, doublets),
then normalization, Harmony integration, and the Mic1 / ExN10_L46 DEG sanity
checks. Each expensive step gets its own checkpoint.